# **spaCy for Text Processing**

## What is spaCy?

[spaCy](https://spacy.io/) is an open-source library for advanced Natural Language Processing (NLP) in Python. It provides a wide range of tools and features for processing and analyzing text data, making it a popular choice among developers and researchers in the field of NLP.

In this session, we will explore the basics of spaCy and how to use it for text processing tasks

## Why spaCy?

Python provides the ability to manipulate text data using built-in string methods and with the helps of library like `pandas` in our previous session. However, these methods and libraries are not designed specifically for NLP tasks and may not be efficient or effective for processing large volumes of text data.

Some tasks that might be hard to do without specialized NLP libraries include:
- Tokenization
- Lemmatization
- Part-of-speech tagging
- Named entity recognition
- And many more...

## Installing spaCy

Although spaCy in Colab is pre-installed, if for some reason you need to install it or install a specific version or update it, you can use the following command:

```bash
!pip install spacy
```

In [22]:
!pip install spacy

### Models & Language Support

spaCy able to process text in multiple languages, and it provides pre-trained models for various languages. You can download and use these models to perform NLP tasks in different languages. For example, to download the English model, you can use the following command:

```bash
!python -m spacy download en_core_web_sm
```

You can see all supported languages and models on the [spaCy models page](https://spacy.io/models).

In [23]:
# In this session we will be using model for english language
MODEL_NAME = "en_core_web_sm"

!python -m spacy download $MODEL_NAME

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 28.6 MB/s  0:00:00 29.1 MB/s eta 0:00:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


## Loading the Model

To load the spaCy model, you can use the following code:

```python
import spacy

nlp = spacy.load(MODEL_NAME)
``` 

This `nlp` object is a spaCy pipeline that you can use to process text data. You can pass a string of text to the `nlp` object, and it will return a `Doc` object that contains the processed text and its annotations.

In [24]:
import spacy

nlp = spacy.load(MODEL_NAME)

paragraph = """
Using spaCy its possible to do NLP tasks easily and efficiently.
This is a simple paragraph to demonstrate how to use spaCy for text processing.
Passing this paragraph to the `nlp` object will return a `Doc` object that contains the processed text and its annotations.
"""

doc = nlp(paragraph)
print("Result type:", type(doc))
for idx, sent in enumerate(doc.sents):
    print(f"Sentence {idx + 1}: {sent.text.strip()}")

Result type: <class 'spacy.tokens.doc.Doc'>
Sentence 1: Using spaCy its possible to do NLP tasks easily and efficiently.
Sentence 2: This is a simple paragraph to demonstrate how to use spaCy for text processing.
Sentence 3: Passing this paragraph to the `nlp` object will return a `Doc` object that contains the processed text and its annotations.


## Text Processing

In [25]:
## No need to concern about this function
## This is helper for getting the text of Alice in Wonderland from Project Gutenberg

import requests
from functools import cache
import pandas as pd

ALICE_URL = "https://www.gutenberg.org/files/11/11-0.txt"


@cache
def get_alice() -> pd.DataFrame:
    # fetch the book from the URL
    book = requests.get(ALICE_URL).text

    # remove the table of contents
    chapters = book.split("CHAPTER ")[13:]

    # content format is [CHAPTER NUMBER].\n[TITLE]\n\n\\n[CONTENT]
    records = []
    for chapter in chapters:
        number, title, content = chapter.split("\n", 2)
        records.append((number.strip(), title.strip(), content.strip()))

    return pd.DataFrame(records, columns=["chapter", "title", "content"])


The function above is helper function to fetch the text of "Alice's Adventures in Wonderland" from Project Gutenberg, split it into chapters, and return a DataFrame with the chapter number, title, and content.

We'll use the book as our sample text data to demonstrate how to use spaCy for text processing tasks.

In [26]:
alice = get_alice()
alice

,chapter,title,content
0,I.,Down the Rabbit-Hole,Alice was beginning to get very tired of sitti...
1,II.,The Pool of Tears,“Curiouser and curiouser!” cried Alice (she wa...
2,III.,A Caucus-Race and a Long Tale,They were indeed a queer-looking party that as...
3,IV.,The Rabbit Sends in a Little Bill,"It was the White Rabbit, trotting slowly back ..."
4,V.,Advice from a Caterpillar,The Caterpillar and Alice looked at each other...
5,VI.,Pig and Pepper,For a minute or two she stood looking at the h...
6,VII.,A Mad Tea-Party,There was a table set out under a tree in fron...
7,VIII.,The Queen’s Croquet-Ground,A large rose-tree stood near the entrance of t...
8,IX.,The Mock Turtle’s Story,“You can’t think how glad I am to see you agai...
9,X.,The Lobster Quadrille,"The Mock Turtle sighed deeply, and drew the ba..."


Python [string](https://docs.python.org/3/library/string.html) has many built-in capabilities. Some common string operations include manipulation such as:
- `str.lower()`
- `str.upper()`
- `str.strip()`
- `str.replace()`
- and many more...

Let's use the 1st chapter of the book as our sample text data.

In [27]:
chapter_1_title = alice.iloc[0]["title"]

print("Chapter title: ", chapter_1_title)
print("Title lowercase: ", chapter_1_title.lower())
print("Title uppercase: ", chapter_1_title.upper())
print("Replace 'Down' with 'Up': ", chapter_1_title.replace("Down", "Up"))

# strip() removes leading and trailing whitespace characters
title_with_whitespace = "   " + chapter_1_title + "   "
print(f"Title with whitespace: '{title_with_whitespace}'")
print(f"Title after strip: '{title_with_whitespace.strip()}'")

Chapter title:  Down the Rabbit-Hole
Title lowercase:  down the rabbit-hole
Title uppercase:  DOWN THE RABBIT-HOLE
Replace 'Down' with 'Up':  Up the Rabbit-Hole
Title with whitespace: '   Down the Rabbit-Hole   '
Title after strip: 'Down the Rabbit-Hole'


For further processing tasks, we want to use the chapter content for our feature. But before that, the text need to be cleaned and preprocessed. This is where spaCy comes in handy.

We can just pass the chapter content to the `nlp` object, and it will return a [`Doc` object](https://spacy.io/api/doc) that contains the processed text and its annotations. We can then use the attributes of the `Doc` object to access the processed text and its annotations.

In [28]:
chapter_1 = alice.iloc[0]["content"]
doc = nlp(chapter_1)

### Tokenization

Tokenization is the process of breaking down a text into smaller units called tokens. Tokens can be words, punctuation marks, or even whitespace. Tokenization is a fundamental step in NLP tasks, as it allows us to analyze and manipulate text at a more granular level.

The `doc` object returned by the `nlp` pipeline contains a list of tokens that we can access. Its as easy as that!

In [29]:
# print the first 10 tokens
for token in doc[:10]:
    print(token.text)

Alice
was
beginning
to
get
very
tired
of
sitting
by


In more advanced use case, we want to split the text into sentences rather than tokens.

Like the example above, we can access the sentences in the `doc` object using the `sents` attribute. This will return a generator of [`Span` objects](https://spacy.io/api/span), which represent the sentences in the text. We can then iterate over the sentences and print them out.

In [30]:
# print the first 5 sentences
for idx, sent in enumerate(doc.sents):
    print(f"Sentence {idx + 1}: {sent.text.strip()}")
    if idx == 4:
        break

Sentence 1: Alice was beginning to get very tired of sitting by her sister on the
bank, and of having nothing to do: once or twice she had peeped into
the book her sister was reading, but it had no pictures or
conversations in it, “and what is the use of a book,” thought Alice
“without pictures or conversations?”
Sentence 2: So she was considering in her own mind (as well as she could, for the
hot day made her feel very sleepy and stupid), whether the pleasure of
making a daisy-chain would be worth the trouble of getting up and
picking the daisies, when suddenly a White Rabbit with pink eyes ran
close by her.
Sentence 3: There was nothing so _very_ remarkable in that; nor did Alice think it
so _very_ much out of the way to hear the Rabbit say to itself, “Oh
dear!
Sentence 4: Oh dear!
Sentence 5: I shall be late!”


In above example, in the sentence there are case when newline character is included. spaCy was able to detect that this newline is still part of the sentence, and it is not a sentence boundary. This is one of the advantages of using spaCy for text processing, as it can handle various edge cases and nuances in natural language.

### Lexical Attributes

The [`Doc` object](https://spacy.io/api/doc) in addition to contain the tokens and sentences, it also contains various lexical attributes that provide information about the tokens in the text or word level attributes. These attributes include:
- `i`: The index of the token in the original text.
- `text`: The original text of the token.
- `lemma_`: The base form of the token (e.g., "running" -> "run").
- `is_stop`: A boolean indicating whether the token is a stop word
- `is_punct`: A boolean indicating whether the token is a punctuation mark.
- `is_space`: A boolean indicating whether the token is a whitespace character.
- `shape_`: The shape of the token (e.g., "Xxxx" for "Alice", "xxxx" for "in", etc.).
- `pos_`: The part-of-speech tag of the token (e.g., "NOUN", "VERB", etc.).
- `tag_`: The detailed part-of-speech tag of the token (e.g., "NN", "VBZ", etc.).
- `dep_`: The syntactic dependency relation of the token (e.g., "nsubj", "dobj", etc.).
- `ent_type_`: The named entity type of the token (e.g., "PERSON", "ORG", etc.).
- and many more...

In [31]:
attributes = []
for token in doc:
    attributes.append(
        {
            "idx": token.i,
            "text": token.text,
            "lemma": token.lemma_,
            "is_stop": token.is_stop,
            "is_punct": token.is_punct,
            "is_space": token.is_space,
            "shape": token.shape_,
            "pos": token.pos_,
            "tag": token.tag_,
            "dep": token.dep_,
            "ent": token.ent_type_,
        }
    )

attributes_df = pd.DataFrame(attributes)
attributes_df.head(10)

,idx,text,lemma,is_stop,is_punct,is_space,shape,pos,tag,dep,ent
0,0,Alice,Alice,False,False,False,Xxxxx,PROPN,NNP,nsubj,PERSON
1,1,was,be,True,False,False,xxx,AUX,VBD,aux,
2,2,beginning,begin,False,False,False,xxxx,VERB,VBG,ROOT,
3,3,to,to,True,False,False,xx,PART,TO,aux,
4,4,get,get,True,False,False,xxx,VERB,VB,xcomp,
5,5,very,very,True,False,False,xxxx,ADV,RB,advmod,
6,6,tired,tired,False,False,False,xxxx,ADJ,JJ,acomp,
7,7,of,of,True,False,False,xx,ADP,IN,prep,
8,8,sitting,sit,False,False,False,xxxx,VERB,VBG,pcomp,
9,9,by,by,True,False,False,xx,ADP,IN,prep,


### Punctuation and Stop Words

Now that we have the attributes of the tokens, we can use them to filter out punctuation and stop words from our text data. This is a common preprocessing step in NLP tasks, as punctuation and stop words often do not carry much meaning and can be removed to reduce noise in the data.

This can be easily done by using the `is_punct` and `is_stop` attributes of the tokens. We can iterate over the tokens in the `Doc` object and create a new list of tokens that only includes those that are not punctuation or stop words.

Here we also use the `lemma_` attribute to get the base form of the tokens, which can help to reduce the dimensionality of the data and improve the performance of NLP models.

In [32]:
cleaned_chapter_1 = []
for token in doc:
    if not token.is_stop and not token.is_punct:
        cleaned_chapter_1.append(token.lemma_)

print("Cleaned Chapter 1 Tokens: ", cleaned_chapter_1[:10])

Cleaned Chapter 1 Tokens:  ['Alice', 'begin', 'tired', 'sit', 'sister', '\n', 'bank', 'have', 'twice', 'peep']


Looks like there are still unwanted tokens like `\n` in the cleaned tokens. We can filter them out by checking the `is_space` attribute of the tokens, which indicates whether the token is a whitespace character. We can modify our code to also exclude tokens that are whitespace characters.

In [33]:
cleaned_chapter_1 = []
for token in doc:
    if not token.is_stop and not token.is_punct and not token.is_space:
        cleaned_chapter_1.append(token.lemma_)

print("Cleaned Chapter 1 Tokens: ", cleaned_chapter_1[:10])

Cleaned Chapter 1 Tokens:  ['Alice', 'begin', 'tired', 'sit', 'sister', 'bank', 'have', 'twice', 'peep', 'book']


## Processing Pipeline

Using all of above, we can create a processing pipeline that takes in raw text data, processes it using spaCy, and returns a cleaned and preprocessed version of the text that is ready for analysis or modeling. This pipeline can be easily applied to large volumes of text data, making it a powerful tool for NLP tasks.

For example we want to transform the chapter content into a cleaned version of the text that only includes the lemmas of the tokens that are not stop words, punctuation, or whitespace characters and lowercase the tokens. We can create a function that takes in the raw text data, processes it using spaCy, and returns the cleaned version of the text.

In [ ]:
def preprocess_text(text):
    doc = nlp(text)
    cleaned_tokens = []
    for token in doc:
        if not token.is_stop and not token.is_punct and not token.is_space:
            cleaned_tokens.append(token.lemma_.lower())

    # join the cleaned tokens back into a single string
    return " ".join(cleaned_tokens)

# Apply the preprocessing function to the content of each chapter in the DataFrame
alice["cleaned_content"] = alice["content"].apply(preprocess_text)

alice

,chapter,title,content,cleaned_content
0,I.,Down the Rabbit-Hole,Alice was beginning to get very tired of sitti...,alice begin tired sit sister bank have twice p...
1,II.,The Pool of Tears,“Curiouser and curiouser!” cried Alice (she wa...,curiouser curiouser cry alice surprised moment...
2,III.,A Caucus-Race and a Long Tale,They were indeed a queer-looking party that as...,queer look party assemble bank bird draggle fe...
3,IV.,The Rabbit Sends in a Little Bill,"It was the White Rabbit, trotting slowly back ...",white rabbit trot slowly look anxiously go los...
4,V.,Advice from a Caterpillar,The Caterpillar and Alice looked at each other...,caterpillar alice look time silence caterpilla...
5,VI.,Pig and Pepper,For a minute or two she stood looking at the h...,minute stand look house wonder suddenly footma...
6,VII.,A Mad Tea-Party,There was a table set out under a tree in fron...,table set tree house march hare hatter have te...
7,VIII.,The Queen’s Croquet-Ground,A large rose-tree stood near the entrance of t...,large rose tree stand near entrance garden ros...
8,IX.,The Mock Turtle’s Story,“You can’t think how glad I am to see you agai...,think glad dear old thing say duchess tuck arm...
9,X.,The Lobster Quadrille,"The Mock Turtle sighed deeply, and drew the ba...",mock turtle sigh deeply draw flapper eye look ...


Having the lexical properties of the text provided by spaCy, we can manipulate and process the text data in various ways to prepare it for further use in NLP tasks. These capabilities make spaCy a powerful and versatile library for text processing and analysis in Python.